[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/06-model-evaluation.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/06-model-evaluation.ipynb)

# Module 5 — Lesson 6: Model Evaluation

**Module:** 5 — Machine Learning Foundations | **Time:** 35 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Construct and interpret confusion matrices
- Calculate and trade off precision, recall, F1, and accuracy
- Plot ROC curves and compute AUC scores
- Draw precision-recall curves for imbalanced classification problems
- Apply k-fold and stratified k-fold cross-validation correctly
- Handle class imbalance with `class_weight='balanced'`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     cross_validate, KFold, StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, precision_score, recall_score,
                             f1_score, accuracy_score, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded.')

## 1. Confusion Matrix

A confusion matrix tabulates predicted versus actual class labels:

```
                  Predicted
                  Negative  Positive
Actual Negative:    TN        FP
Actual Positive:    FN        TP
```

- **True Positive (TP)** — correctly predicted positive
- **False Positive (FP)** — predicted positive, actually negative (Type I error)
- **False Negative (FN)** — predicted negative, actually positive (Type II error)
- **True Negative (TN)** — correctly predicted negative

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = Pipeline([
    ('sc', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
print('\nBreakdown:')
tn, fp, fn, tp = cm.ravel()
print(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, norm, title in zip(axes, [None, 'true'], ['Raw Counts', 'Normalised (row)']):
    cm_plot = confusion_matrix(y_test, y_pred, normalize=norm)
    disp = ConfusionMatrixDisplay(cm_plot, display_labels=data.target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {title}')
plt.tight_layout()
plt.show()

## 2. Precision, Recall, F1 — Tradeoffs

| Metric | Formula | Focus |
|---|---|---|
| **Accuracy** | (TP+TN) / N | Overall correctness — misleading for imbalanced data |
| **Precision** | TP / (TP+FP) | Of all predicted positives, how many are correct? |
| **Recall (Sensitivity)** | TP / (TP+FN) | Of all actual positives, how many did we catch? |
| **F1-Score** | 2·P·R / (P+R) | Harmonic mean of precision and recall |

Adjusting the **decision threshold** moves the tradeoff between precision and recall.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 50)
precisions, recalls, f1_scores, accuracies = [], [], [], []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))
    accuracies.append(accuracy_score(y_test, y_pred_t))

best_f1_thresh = thresholds[np.argmax(f1_scores)]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precisions,  label='Precision',  color='steelblue', lw=2)
plt.plot(thresholds, recalls,     label='Recall',     color='tomato',    lw=2)
plt.plot(thresholds, f1_scores,   label='F1-Score',   color='green',     lw=2)
plt.plot(thresholds, accuracies,  label='Accuracy',   color='purple',    lw=2, linestyle='--')
plt.axvline(best_f1_thresh, color='black', linestyle=':', label=f'Best F1 threshold={best_f1_thresh:.2f}')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Decision Threshold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'At default threshold 0.5:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 3. ROC Curve and AUC

The ROC (Receiver Operating Characteristic) curve plots **True Positive Rate** (Recall) vs **False Positive Rate** at all possible thresholds.

- A random classifier sits on the diagonal (AUC = 0.5)
- A perfect classifier has AUC = 1.0
- AUC is threshold-independent and useful for comparing models

In [ ]:
# Compare multiple models on ROC
models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

plt.figure(figsize=(8, 6))
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC=0.5)')
plt.fill_between([0, 1], [0, 1], alpha=0.05, color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — Breast Cancer Dataset')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Precision-Recall Curve for Imbalanced Data

For heavily imbalanced datasets, the ROC-AUC can be misleadingly optimistic. The **Precision-Recall curve** focuses on the positive (minority) class only. The **Average Precision (AP)** score summarises it as a single number.

In [ ]:
# Create an imbalanced dataset (5% positive)
X_imb, y_imb = make_classification(n_samples=2000, n_features=20, n_informative=10,
                                    weights=[0.95, 0.05], random_state=42)
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_imb, y_imb, test_size=0.2,
                                                    stratify=y_imb, random_state=42)
print(f'Class distribution (test): {np.bincount(y_te_i)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, use_balanced) in zip(axes, [('Default (no balancing)', False),
                                            ('class_weight="balanced"', True)]):
    cw = 'balanced' if use_balanced else None
    lr_imb = Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(class_weight=cw, max_iter=1000))
    ])
    lr_imb.fit(X_tr_i, y_tr_i)
    probs_i = lr_imb.predict_proba(X_te_i)[:, 1]

    prec, rec, _ = precision_recall_curve(y_te_i, probs_i)
    ap = average_precision_score(y_te_i, probs_i)
    auc_r = roc_auc_score(y_te_i, probs_i)

    ax.step(rec, prec, where='post', lw=2, color='steelblue')
    ax.fill_between(rec, prec, alpha=0.15, color='steelblue')
    ax.axhline(y_te_i.mean(), linestyle='--', color='red', label=f'Baseline ({y_te_i.mean():.2f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{name}\nAP={ap:.3f}  |  AUC-ROC={auc_r:.3f}')
    ax.legend()

plt.suptitle('Precision-Recall Curves — Imbalanced Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. K-Fold vs Stratified K-Fold Cross-Validation

Stratified K-Fold ensures each fold preserves the class proportion. This is especially important for imbalanced datasets.

In [ ]:
pipe_cv = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kf_scores  = cross_val_score(pipe_cv, X, y, cv=kf,  scoring='accuracy')
skf_scores = cross_val_score(pipe_cv, X, y, cv=skf, scoring='accuracy')

print('KFold          scores:', kf_scores.round(4), f'  mean={kf_scores.mean():.4f}')
print('StratifiedKFold scores:', skf_scores.round(4), f'  mean={skf_scores.mean():.4f}')

# cross_validate for multiple metrics simultaneously
cv_results = cross_validate(
    pipe_cv, X, y,
    cv=skf,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=True
)
print('\ncross_validate — Test Results (5-Fold Stratified):')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    test_m = cv_results[f'test_{metric}']
    print(f'  {metric:12s}: {test_m.mean():.4f} ± {test_m.std():.4f}')

## 6. Full Evaluation Summary — Metrics Dashboard

In [ ]:
final_models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []
for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_p = model.predict(X_test)
    y_pr = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, y_p), 4),
        'Precision': round(precision_score(y_test, y_p), 4),
        'Recall':    round(recall_score(y_test, y_p), 4),
        'F1':        round(f1_score(y_test, y_p), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, y_pr), 4)
    })

results_df = pd.DataFrame(results)
print('=== Model Evaluation Dashboard ===')
print(results_df.to_string(index=False))

# Bar chart dashboard
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
x = np.arange(len(metrics))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
for i, row in results_df.iterrows():
    ax.bar(x + i*width, [row[m] for m in metrics], width, label=row['Model'], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0.9, 1.01)
ax.set_ylabel('Score')
ax.set_title('Model Evaluation Dashboard')
ax.legend()
plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — Threshold Optimisation for Recall**
In medical screening (like cancer detection) we often prefer high recall over high precision. Using the logistic regression model from section 1, find the decision threshold that achieves recall >= 0.98 while maximising precision. Print the threshold, precision, recall, and F1 score at that threshold.

**Exercise 2 — Imbalanced Dataset Full Evaluation**
Using the imbalanced dataset from section 4 (weights=[0.95, 0.05]), train a `RandomForestClassifier` with and without `class_weight='balanced'`. Compare both models using: confusion matrix, precision, recall, F1, ROC-AUC, and average precision. Which metric shows the biggest improvement from balancing?

**Exercise 3 — Cross-Validation Stability Analysis**
Run 5-fold, 10-fold, and Leave-One-Out (LOO) cross-validation on `load_breast_cancer()` using a `LogisticRegression` pipeline. Compare the mean accuracy and standard deviation across folds for each strategy. Comment on the trade-off between computational cost and estimate stability.